# Statystyka dla Data Analyst/Engineer — Moduł 3: Podstawy prawdopodobieństwa

Zanim przejdziemy do rozkładów prawdopodobieństwa (Moduł 4) i testów statystycznych
(Moduły 7+), potrzebujemy wspólnego języka: czym w ogóle jest prawdopodobieństwo, jak się
je liczy, i co się dzieje, gdy zdarzenia są od siebie zależne. Ten moduł jest bardziej
teoretyczny niż poprzednie, ale każde pojęcie od razu policzymy zarówno "z definicji", jak
i empirycznie na naszych danych klientów.

## Spis treści
1. [Czym jest prawdopodobieństwo?](#sec1)
2. [Podstawowe reguły: dopełnienie, suma, iloczyn](#sec2)
3. [Prawdopodobieństwo warunkowe](#sec3)
4. [Niezależność zdarzeń](#sec4)
5. [Twierdzenie Bayesa](#sec5)
6. [Prawo wielkich liczb — symulacja](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Czym jest prawdopodobieństwo?

**Prawdopodobieństwo** zdarzenia to liczba z przedziału `[0, 1]` (albo procent od 0% do
100%) mówiąca, jak często to zdarzenie zachodzi. `0` oznacza "nigdy", `1` oznacza "zawsze".
**Przestrzeń zdarzeń elementarnych** to zbiór wszystkich możliwych wyników — np. dla rzutu
kostką sześcienną to `{1, 2, 3, 4, 5, 6}`. **Zdarzenie** to dowolny podzbiór tej
przestrzeni, np. "wypadła liczba parzysta" = `{2, 4, 6}`.

Najprostszy sposób liczenia prawdopodobieństwa (dla wyników równo prawdopodobnych):

```
P(zdarzenie) = liczba wynikow sprzyjajacych / liczba wszystkich mozliwych wynikow
```

In [ ]:
# P(wypadnie parzysta liczba oczek) przy rzucie kostka
wyniki_sprzyjajace = 3  # {2, 4, 6}
wszystkie_wyniki = 6  # {1, 2, 3, 4, 5, 6}
p_parzysta = wyniki_sprzyjajace / wszystkie_wyniki
print(p_parzysta)


To jest podejście *klasyczne* (z definicji, bez eksperymentu). Możemy też liczyć
prawdopodobieństwo *empirycznie* — z częstości występowania w danych, które już mamy.
Np. jakie jest prawdopodobieństwo, że losowo wybrany klient jest z Warszawy?

In [ ]:
p_warszawa = (klienci["miasto"] == "Warszawa").mean()
print(p_warszawa)


> 📊 **mean() na warunku logicznym to trik warty zapamiętania**
>
> `(klienci["miasto"] == "Warszawa")` daje Series wartości `True`/`False`. `pandas` traktuje `True` jako `1`, a `False` jako `0`, więc `.mean()` tej Series to dokładnie odsetek wierszy spełniających warunek — czyli empiryczne prawdopodobieństwo. Zobaczysz ten trik wielokrotnie w dalszych modułach.

<a id="sec2"></a>
## 2. Podstawowe reguły: dopełnienie, suma, iloczyn

- **Dopełnienie**: `P(nie A) = 1 - P(A)`.
- **Reguła sumy** (dla zdarzeń, które mogą zachodzić razem): `P(A lub B) = P(A) + P(B) -
  P(A i B)`. Odejmujemy `P(A i B)`, żeby nie policzyć części wspólnej dwa razy.
- **Reguła iloczynu** (dla zdarzeń **niezależnych**): `P(A i B) = P(A) * P(B)`. O
  niezależności więcej w sekcji 4 — na razie zakładamy, że tak jest.

In [ ]:
p_gdansk = (klienci["miasto"] == "Gdansk").mean()
p_konwersja = (klienci["konwersja"] == 1).mean()

# P(miasto Warszawa LUB miasto Gdansk) -- zdarzenia sie WYKLUCZAJA (klient ma jedno miasto),
# wiec P(A i B) = 0 i regula sumy upraszcza sie do zwyklego dodawania
p_warszawa_lub_gdansk = p_warszawa + p_gdansk
print(f"P(Warszawa lub Gdansk): {p_warszawa_lub_gdansk:.3f}")

# P(miasto Warszawa) * P(konwersja) -- TRAKTUJEMY jako niezalezne (zalozenie, nie fakt!)
p_warszawa_i_konwersja_zakladajac_niezaleznosc = p_warszawa * p_konwersja
print(f"P(Warszawa) * P(konwersja) przy zalozeniu niezaleznosci: {p_warszawa_i_konwersja_zakladajac_niezaleznosc:.3f}")


> ⚠️ **Reguła sumy dla zdarzeń, które NIE wykluczają się wzajemnie**
>
> Powyższy przykład Warszawa/Gdańsk działa prosto, bo klient nie może być jednocześnie "z Warszawy" i "z Gdańska" — te zdarzenia się **wykluczają** (`P(A i B) = 0`). Ale np. "klient jest z Warszawy" i "klient dokonał konwersji" **nie wykluczają się** — trzeba by odjąć `P(Warszawa i konwersja)`, którą policzymy naprawdę (bez zakładania niezależności) w sekcji 4.

<a id="sec3"></a>
## 3. Prawdopodobieństwo warunkowe

**Prawdopodobieństwo warunkowe** `P(A|B)` (czytaj: "prawdopodobieństwo A, pod warunkiem
że B") to prawdopodobieństwo zdarzenia A, gdy już wiemy, że zaszło B. Wzór:

```
P(A|B) = P(A i B) / P(B)
```

Innymi słowy: zawężamy naszą "przestrzeń możliwości" tylko do przypadków, w których B
jest prawdą, i patrzymy, jaki odsetek z nich to też A.

In [ ]:
# P(konwersja | grupa B) -- jaki jest wspolczynnik konwersji WSROD klientow z grupy B?
grupa_b = klienci[klienci["grupa_kampanii"] == "B"]
p_konwersja_pod_b = (grupa_b["konwersja"] == 1).mean()

# to samo, wprost ze wzoru P(A i B) / P(B)
p_konwersja_i_b = ((klienci["grupa_kampanii"] == "B") & (klienci["konwersja"] == 1)).mean()
p_b = (klienci["grupa_kampanii"] == "B").mean()
p_konwersja_pod_b_ze_wzoru = p_konwersja_i_b / p_b

print(f"P(konwersja | grupa B) bezposrednio: {p_konwersja_pod_b:.3f}")
print(f"P(konwersja | grupa B) ze wzoru: {p_konwersja_pod_b_ze_wzoru:.3f}")


Obie metody dają ten sam wynik — to jest dokładnie ten sam efekt A/B, który zobaczyłeś/
aś wizualnie w Module 2 i który sformalizujemy testem statystycznym w Module 9
(chi-kwadrat).

<a id="sec4"></a>
## 4. Niezależność zdarzeń

Zdarzenia A i B są **niezależne**, gdy zajście jednego nie zmienia prawdopodobieństwa
drugiego: `P(A|B) = P(A)` (równoważnie: `P(A i B) = P(A) * P(B)`). Jeśli `P(A|B) != P(A)`,
zdarzenia są **zależne** — wiedza o B faktycznie coś nam mówi o A.

In [ ]:
p_konwersja_ogolem = (klienci["konwersja"] == 1).mean()
p_konwersja_pod_a = (klienci[klienci["grupa_kampanii"] == "A"]["konwersja"] == 1).mean()
p_konwersja_pod_b = (klienci[klienci["grupa_kampanii"] == "B"]["konwersja"] == 1).mean()

print(f"P(konwersja) ogolem: {p_konwersja_ogolem:.3f}")
print(f"P(konwersja | grupa A): {p_konwersja_pod_a:.3f}")
print(f"P(konwersja | grupa B): {p_konwersja_pod_b:.3f}")


`P(konwersja | grupa A)` i `P(konwersja | grupa B)` wyraźnie różnią się od
`P(konwersja)` ogółem (i od siebie nawzajem) — konwersja i grupa kampanii **nie są
niezależne**. To dokładnie oczekiwany wynik: przecież sami zaprojektowaliśmy dane tak,
żeby grupa B miała wyższą konwersję.

<a id="sec5"></a>
## 5. Twierdzenie Bayesa

**Twierdzenie Bayesa** pozwala "odwrócić" prawdopodobieństwo warunkowe — z `P(B|A)`
policzyć `P(A|B)`:

```
P(A|B) = P(B|A) * P(A) / P(B)
```

Klasyczny przykład: znamy `P(pozytywny wynik testu | choroba)` (czułość testu), ale
naprawdę interesuje nas `P(choroba | pozytywny wynik testu)`. Policzmy analogiczne pytanie
na naszych danych: **mając klienta, który dokonał konwersji, jakie jest
prawdopodobieństwo, że był w grupie B?**

In [ ]:
# Znamy juz (z sekcji 3-4):
p_b = (klienci["grupa_kampanii"] == "B").mean()               # P(B)
p_konwersja_pod_b = (klienci[klienci["grupa_kampanii"] == "B"]["konwersja"] == 1).mean()  # P(konwersja | B)
p_konwersja = (klienci["konwersja"] == 1).mean()               # P(konwersja)

# Bayes: P(B | konwersja) = P(konwersja | B) * P(B) / P(konwersja)
p_b_pod_konwersja = p_konwersja_pod_b * p_b / p_konwersja
print(f"P(grupa B | konwersja): {p_b_pod_konwersja:.3f}")

# Weryfikacja bezposrednio z danych
przekonwertowani = klienci[klienci["konwersja"] == 1]
p_b_pod_konwersja_bezposrednio = (przekonwertowani["grupa_kampanii"] == "B").mean()
print(f"To samo, policzone wprost: {p_b_pod_konwersja_bezposrednio:.3f}")


> 📊 **Twierdzenie Bayesa jest wszędzie w uczeniu maszynowym**
>
> Filtry antyspamowe, klasyfikatory Naive Bayes, systemy rekomendacji, diagnostyka medyczna — wszystkie te zastosowania w gruncie rzeczy odwracają znaną nam "czułość" (P(objaw|przyczyna)) do interesującego nas "co naprawdę się dzieje" (P(przyczyna|objaw)). To jedno z najważniejszych pojęć w całej statystyce.

<a id="sec6"></a>
## 6. Prawo wielkich liczb — symulacja

**Prawo wielkich liczb** mówi, że im więcej powtórzeń losowego eksperymentu, tym bliżej
empiryczna częstość zbliża się do prawdziwego prawdopodobieństwa teoretycznego.
Zasymulujmy rzuty monetą (P(orzeł) = 0.5 z definicji) i zobaczmy, jak częstość orłów
stabilizuje się przy rosnącej liczbie rzutów.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
liczba_rzutow = 2000
rzuty = rng.integers(0, 2, size=liczba_rzutow)  # 0 = reszka, 1 = orzel

skumulowana_czestosc_orla = np.cumsum(rzuty) / np.arange(1, liczba_rzutow + 1)

plt.figure(figsize=(8, 4))
plt.plot(skumulowana_czestosc_orla)
plt.axhline(0.5, color="red", linestyle="--", label="Prawdziwe P(orzel) = 0.5")
plt.xlabel("Liczba rzutow")
plt.ylabel("Skumulowana czestosc orla")
plt.title("Prawo wielkich liczb: rzuty moneta")
plt.legend()
plt.show()


Przy niewielkiej liczbie rzutów częstość mocno "skacze" (jeden dodatkowy orzeł robi
dużą różnicę procentową), ale im więcej rzutów, tym linia stabilizuje się coraz bliżej
`0.5`. To właśnie dlatego w Module 5 (Centralne Twierdzenie Graniczne) większe próbki dają
bardziej wiarygodne oszacowania niż małe.

<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- klasyczną i empiryczną definicję prawdopodobieństwa,
- reguły dopełnienia, sumy i iloczynu,
- prawdopodobieństwo warunkowe `P(A|B)`,
- niezależność zdarzeń i jak ją sprawdzić,
- twierdzenie Bayesa do "odwracania" prawdopodobieństwa warunkowego,
- prawo wielkich liczb jako intuicję stojącą za wiarygodnością większych próbek.

> 📝 **Ćwiczenie 1: Prawdopodobieństwo z dwóch kostek**
>
> Zasymuluj (kodem, nie ręcznie) 100 000 rzutów DWIEMA kostkami sześciennymi (`rng.integers(1, 7, size=(100000, 2))`), a następnie policz empiryczne prawdopodobieństwo, że suma oczek wynosi dokładnie 7. Porównaj z wartością teoretyczną `6/36`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
rng2 = np.random.default_rng(1)
rzuty_dwoch_kostek = rng2.integers(1, 7, size=(100_000, 2))
suma = rzuty_dwoch_kostek.sum(axis=1)
p_siedem = (suma == 7).mean()

print(f"Empiryczne P(suma=7): {p_siedem:.4f}")
print(f"Teoretyczne P(suma=7): {6/36:.4f}")
```
</details>

> 📝 **Ćwiczenie 2: Warunkowa satysfakcja**
>
> Policz `P(satysfakcja >= 7 | grupa_kampanii == 'B')` oraz `P(satysfakcja >= 7 | grupa_kampanii == 'A')`. Czy grupa kampanii wydaje się mieć związek z wysoką satysfakcją, czy to raczej przypadek (dane nie zostały tak zaprojektowane)?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
p_wysoka_pod_b = (klienci[klienci["grupa_kampanii"] == "B"]["satysfakcja"] >= 7).mean()
p_wysoka_pod_a = (klienci[klienci["grupa_kampanii"] == "A"]["satysfakcja"] >= 7).mean()
print(p_wysoka_pod_b, p_wysoka_pod_a)
```

Satysfakcja zalezy w naszych danych od wydatkow i regionu, NIE od grupy kampanii -- oczekuj, ze te dwie wartosci beda do siebie zblizone (ewentualna roznica to przypadkowy szum, nie prawdziwy efekt).
</details>

> 📝 **Ćwiczenie 3: Niezależność miasta i regionu**
>
> Sprawdź, czy `P(region == 'Zachod' | miasto == 'Warszawa')` różni się od `P(region == 'Zachod')` ogółem. Czy region i miasto są w naszych danych niezależne? (Podpowiedź: w kodzie generującym dane `region` losowany jest zupełnie osobno od `miasto`.)

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
p_zachod_pod_warszawa = (klienci[klienci["miasto"] == "Warszawa"]["region"] == "Zachod").mean()
p_zachod_ogolem = (klienci["region"] == "Zachod").mean()
print(p_zachod_pod_warszawa, p_zachod_ogolem)
```

Powinny wyjsc bardzo blisko siebie (drobne roznice to tylko szum próbki) -- region i miasto sa w kodzie generujacym dane losowane niezaleznie od siebie.
</details>

> 📝 **Ćwiczenie 4: Bayes dla wysokich wydatków**
>
> Twierdzenie Bayesa: policz `P(grupa_kampanii == 'B' | wydatki_miesieczne > 300)` -- czyli, mając klienta o wysokich wydatkach, jakie jest prawdopodobieństwo, że należy do grupy B? Skorzystaj ze wzoru Bayesa (P(B|A) = P(A|B)*P(B)/P(A)) ORAZ sprawdź wynik bezpośrednio z danych.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
p_b = (klienci["grupa_kampanii"] == "B").mean()
p_wysokie_pod_b = (klienci[klienci["grupa_kampanii"] == "B"]["wydatki_miesieczne"] > 300).mean()
p_wysokie = (klienci["wydatki_miesieczne"] > 300).mean()

p_b_pod_wysokie = p_wysokie_pod_b * p_b / p_wysokie
print(f"Ze wzoru Bayesa: {p_b_pod_wysokie:.3f}")

bezposrednio = (klienci[klienci["wydatki_miesieczne"] > 300]["grupa_kampanii"] == "B").mean()
print(f"Bezposrednio z danych: {bezposrednio:.3f}")
```

Wydatki nie zaleza od grupy kampanii w naszych danych, wiec spodziewaj sie wyniku bliskiego P(B) ogolem (ok. 0.5) -- to kolejny przyklad braku zaleznosci.
</details>

> 🔥 **Wyzwanie: paradoks urodzinowy**
>
> Zasymuluj klasyczny "paradoks dnia urodzin": dla grupy 23 losowych osób (dni urodzin losowane równomiernie z 365 dni, `rng.integers(0, 365, size=23)`), jakie jest prawdopodobieństwo, że przynajmniej dwie mają urodziny tego samego dnia? Powtórz eksperyment 10 000 razy (pętla albo wektoryzacja) i policz odsetek powtórzeń, w których wystąpiło dopasowanie. Podpowiedź: `len(set(dni)) < 23` oznacza, że jakieś dwie daty się powtórzyły. Wynik zaskakuje większość ludzi -- sprawdź, czy wyjdzie bliski 50%.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
rng3 = np.random.default_rng(2)
liczba_symulacji = 10_000
trafienia = 0

for _ in range(liczba_symulacji):
    dni_urodzin = rng3.integers(0, 365, size=23)
    if len(set(dni_urodzin)) < 23:
        trafienia += 1

print(f"P(co najmniej dwie osoby maja urodziny tego samego dnia): {trafienia / liczba_symulacji:.3f}")
```

Teoretyczna wartosc to ok. 0.507 -- wynik symulacji powinien byc bardzo blisko.
</details>

## Co dalej?

W **Module 4** przejdziemy od pojedynczych zdarzeń do **rozkładów prawdopodobieństwa** —
matematycznych modeli opisujących całe rodziny zdarzeń losowych: rozkład normalny,
dwumianowy, Poissona i jednostajny. Poznasz `scipy.stats` jako narzędzie do pracy z nimi.